In [1]:
import pandas as pd
from pyBigKinds.preprocessing import tfidf

import os
import math
import warnings
warnings.filterwarnings("ignore")

pd.set_option('display.max_rows', None)
# Anchor absolute paths to this notebook's own location so repeated
# os.chdir() calls below stay correct regardless of machine/user.
MODEL_DIR = os.path.abspath("../../model") + "/"
NOTEBOOK_TOPICS_DIR = os.getcwd()


In [2]:
def load_dataset(name):
    path = MODEL_DIR
    
    if name == "양극성 장애":
        os.chdir(path + "bipolar_disorder")
        topic_df = pd.read_excel("topic_name_bipolar.xlsx", engine="openpyxl")
        topic_df = topic_df[["Group", "Topic Name","Topic"]]

        all_df = pd.read_excel("df_bipolar.xlsx", engine="openpyxl")
        all_df.columns = ["name", "press", "date", "keywords", "year"]

        doc_df = pd.read_excel("doc_topics_bipolar.xlsx", engine="openpyxl")
        doc_df = doc_df[["name", "Document", "Topic"]]
        
        analysis_df = pd.merge(doc_df, all_df[["name", "press", "year"]], how="left")
        analysis_df = analysis_df.drop_duplicates(["name", "Document", "press"])
        
        df = pd.merge(analysis_df, topic_df, on='Topic')
        df["year_range"] = df["year"].apply(lambda x: math.floor(x/10)*10)
        
        
    elif name == "우울증":
        os.chdir(path + "depression")
        topic_df = pd.read_excel("topic_name_depression.xlsx", engine="openpyxl")
        topic_df = topic_df[["Group", "Topic"]]

        all_df = pd.read_excel("df_depression.xlsx", engine="openpyxl")
        all_df.columns = ["name", "press","keywords", "year"]

        doc_df = pd.read_excel("doc_topics_depression.xlsx", engine="openpyxl")
        
        analysis_df = pd.merge(doc_df, all_df[["name", "press", "year"]], how = "left")
        analysis_df = analysis_df.drop_duplicates(["name", "Document", "press"])
        df = pd.merge(analysis_df, topic_df, on='Topic')
        df["year_range"] = df["year"].apply(lambda x: math.floor(x/10)*10)
        
    elif name == "조현병":
        os.chdir(path + "schizophrenia")
        topic_df = pd.read_excel("topic_name_schizo.xlsx", engine="openpyxl")
        topic_df = topic_df[["Group", "Topic"]]

        all_df = pd.read_excel("df_schizo.xlsx", engine="openpyxl")
        all_df.columns = ["name", "press", "date", "keywords", "year"]

        doc_df = pd.read_excel("doc_topics_schizo.xlsx", engine="openpyxl")
        
        analysis_df = pd.merge(doc_df, all_df[["name", "press", "year"]], how = "left")
        analysis_df = analysis_df.drop_duplicates(["name", "Document", "press"])
        
        df = pd.merge(analysis_df, topic_df, on='Topic')
        df["year_range"] = df["year"].apply(lambda x: math.floor(x/10)*10)
        
    return df

In [3]:
bipolar_df = load_dataset("양극성 장애")
depression_df = load_dataset("우울증")
schizo_df = load_dataset("조현병")

In [4]:
pd.DataFrame({
    "양극성 장애 토픽 개수": [len(bipolar_df["Topic"].unique())],
    "조현병 토픽 개수": [len(schizo_df["Topic"].unique())],
    "우울증 토픽 개수": [len(depression_df["Topic"].unique())],
}).style.set_caption("정신질환 별 BERTopic 전체 토픽 개수")

,양극성 장애 토픽 개수,조현병 토픽 개수,우울증 토픽 개수
0,8,63,329


In [5]:
pd.DataFrame({
    "양극성 장애 최종 토픽 개수": [len(bipolar_df["Topic"].unique()) - 1],
    "조현병 토픽 최종 개수": len(schizo_df["Group"].unique()),
    "우울증 토픽 최종 개수": len(depression_df["Group"].unique()),
}).style.set_caption("정신질환 별 병합된 토픽 개수")

,양극성 장애 최종 토픽 개수,조현병 토픽 최종 개수,우울증 토픽 최종 개수
0,7,11,74


In [6]:
def topic_description(df):
    result = pd.DataFrame(columns=["Topic", "Number", "상위 빈도 단어 15개(TF-IDF 기준)"])
    if "Topic Name" in df.columns:
        for cat in df["Topic Name"].unique():
            tmp = df[df["Topic Name"] == cat].copy()
            tmp["키워드"] = tmp["Document"]
            words = tfidf(tmp).head(15)["단어"].tolist()
            
            tmp_df = pd.DataFrame({
                "Topic": [cat], 
                "Number": [len(tmp)], 
                "상위 빈도 단어 15개(TF-IDF 기준)": [','.join(words)]
                })
            result = pd.concat([result, tmp_df]).sort_values("Number", ascending=False).reset_index(drop=True)
    else:
        for cat in df["Group"].unique():
            tmp = df[df["Group"] == cat].copy()
            tmp["키워드"] = tmp["Document"]
            words = tfidf(tmp).head(15)["단어"].tolist()
            
            tmp_df = pd.DataFrame({
                "Topic": [cat], 
                "Number": [len(tmp)], 
                "상위 빈도 단어 15개(TF-IDF 기준)": [','.join(words)]
                })
            result = pd.concat([result, tmp_df]).sort_values("Number", ascending=False).reset_index(drop=True)

    return result

In [7]:
topic_description(bipolar_df[~bipolar_df['Topic Name'].str.contains('손예진')]).style.set_caption("양극성 장애 토픽 이름 및 정보")

,Topic,Number,상위 빈도 단어 15개(TF-IDF 기준)
0,양극성 장애와 사회적 사건,2466,"정신,환자,장애,치료,경찰,사람,병원,우울증,질환,자살,조울증,건강,사회,자신,범행"
1,정치인과 양극성 장애 관련 주제,60,"지사,경찰,검찰,입원,수사,강제,정신,혐의,사건,허위,사실,고발,공표,진단,후보"
2,유진 박 사건,60,"유진박,매니저,장애,센터,유진,양극,천재,혐의,서울,조울증,공연,바이올리니스트,착취,횡령,경찰"
3,장근석 사회복무요원 복무,56,"장근석,복무,사회,장애,양극,요원,시간,병역,대체,입대,판정,무매독자,소속,신체,배우"
4,김승연 회장 구속 집행정지,44,"회장,구속,집행정지,연장,김승연,혐의,선고,계열사,기간,징역,재판부,그룹,배임,항소심,상태"
5,스윙스의 의병 제대,25,"스윙스,제대,가사,개월,복무,정신,현역,치료,복용,정도,질환,스트레스,문지훈,합심,부적합"
6,양극성 장애와 모자보건법,23,"유전,정신,질환,불임수술,낙태,보건,허용,규정,명령,환자,시술,강제,모자,임신,법제"


In [8]:
topic_description(schizo_df).style.set_caption("조현병 토픽 이름 및 정보")

,Topic,Number,상위 빈도 단어 15개(TF-IDF 기준)
0,조현병과 예술 작품,590,"영화,정신,작품,사람,소설,내시,작가,감독,사랑,자신,이야기,드라마,환자,덕혜옹주,그림"
1,조현병과 사회보건,540,"정신,환자,입원,치료,건강,병원,복지,사회,질환,장애,의료,센터,관리,범죄,지원"
2,조현병 환자 범죄 관련 재판 결과,356,"범행,선고,징역,살해,정신,재판,심신,혐의,살인,흉기,피고인,재판부,치료,상태,미약"
3,조현병과 정치적 이슈,345,"대통령,장애,정신,국가,의원,회장,백악관,미국,국민,정부,트럼프,보좌관,정치,사람,청와대"
4,조현병과 사회적 이슈,340,"정신,환자,치료,장애,자살,사람,질환,마약,사회,증상,아이,병원,학생,부모,건강"
5,조현병 관련 의학 연구,328,"유전자,정신,유전,신경,세포,연구,치료,질환,환자,인간,교수,치료제,미국,과학,분열증"
6,조현병과 관련한 사건사고,214,"경찰,학교,학대,정신,아동,친모,치료,혐의,국가,교사,범행,유공자,조사,징역,계부"
7,조현병과 강력범죄,201,"경찰,흉기,병원,정신,환자,어머니,아버지,혐의,조사,범행,아들,경위,살해,치료,조현"
8,조현병과 신체검사,140,"정신,면허,병역,경찰,병역면제,운전,환자,의료,질환,조현,면제,치료,사고,박해진,검사"
9,강남역 살인 사건,105,"여성,혐오,범죄,사건,사회,경찰,남성,정신,살인,범행,강남역,추모,화장실,서울,사람"


In [9]:
topic_description(depression_df).head(15).style.set_caption("조현병 토픽 이름 및 정보 상위 15개")

,Topic,Number,상위 빈도 단어 15개(TF-IDF 기준)
0,우울증과 예술인,1647,"영화,사람,방송,노래,배우,무대,감독,음악,연기,생각,남편,우울증,출연,결혼,공연"
1,우울증을 유발하는 질환,1444,"치매,우울증,수면,치료,환자,장애,사람,정신,증상,우울,스트레스,건강,여성,시간,불면증"
2,우울증과 예술 작품,1016,"소설,작가,작품,사람,그림,교수,문학,여성,사랑,자신,이야기,사회,정신,시인,저자"
3,우울증과 경찰 수사 관련 이슈,915,"경찰,발견,아들,아파트,조사,우울증,신고,자살,살해,범행,투신,가족,자신,병원,경위"
4,우울증과 의약물,871,"치료,마약,병원,환자,건강,치료제,정신,금연,우울증,담배,강좌,교수,질환,의료,센터"
5,우울증과 사회적 트라우마,795,"소방관,정신,환자,사고,장애,치료,사람,병원,교수,지하철,스트레스,참사,세월,자살,외상"
6,우울증과 자연/음식/문화,758,"동물,반려,사람,미세먼지,치유,돌고래,기후,산림,건강,프로그램,마음,치료,식물,인간,정원"
7,정치/경제 정책과 우울증,727,"대통령,경제,의원,정치,사람,후보,투자,사회,국민,주식,미국,정부,민주당,선거,서울"
8,우울증과 폭력/성폭력 사건,669,"교사,학교,학생,폭력,아이,피해자,학부모,사건,교육,아동,경찰,부모,학대,피해,교수"
9,우울증과 노동자 관련 이슈,621,"노동자,노조,감정,쌍용,노동,직장,회사,고객,해고,조사,건강,정신,센터,유성기업,업무"


In [ ]:
os.chdir(NOTEBOOK_TOPICS_DIR)

topic_description(bipolar_df).to_excel("data/bipolar_topics.xlsx", index=False)
topic_description(depression_df).to_excel("data/depression_topics.xlsx", index=False)
topic_description(schizo_df).to_excel("data/schizo_topics.xlsx", index=False)